In [1]:
# MiniGPT — Model Evaluation

# This notebook evaluates the character-level MiniGPT trained on the Tiny Shakespeare dataset.

# We will measure:

#- Training loss
#- Validation loss
#- Perplexity
#- Character prediction accuracy
#- Generation behavior
#- Sampling strategies
#- Model size

# The goal is to understand how well the model learned the statistical structure of the training corpus.

In [2]:
import os
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

In [3]:
print("PyTorch:", torch.__version__)
print("MPS available:", torch.backends.mps.is_available())

PyTorch: 2.14.0
MPS available: True


In [4]:
device = torch.device(
    "mps" if torch.backends.mps.is_available() else "cpu"
)

print("Evaluation device:", device)

Evaluation device: mps


In [5]:
vocab_size = 63
context_length = 128

embedding_dim = 256
num_heads = 4
num_layers = 4

dropout = 0.2

In [6]:
MODEL_PATH = "../models/minigpt_checkpoint.pt"

print("Checkpoint:", MODEL_PATH)
print("Exists:", os.path.exists(MODEL_PATH))

Checkpoint: ../models/minigpt_checkpoint.pt
Exists: True


In [7]:
checkpoint = torch.load(
    MODEL_PATH,
    map_location=device,
    weights_only=False
)

print("Checkpoint loaded.")
print("Iteration:", checkpoint["iteration"])
print("Training loss:", checkpoint["train_loss"])
print("Validation loss:", checkpoint["val_loss"])

Checkpoint loaded.
Iteration: 5000
Training loss: 1.3438644409179688
Validation loss: 1.4844762086868286


In [8]:
stoi = checkpoint["stoi"]
itos = checkpoint["itos"]

print("Vocabulary size:", len(stoi))

Vocabulary size: 63


In [9]:
def encode(text):
    return [
        stoi[c]
        for c in text
    ]


def decode(tokens):
    return "".join(
        itos[token]
        for token in tokens
    )

In [10]:
class Head(nn.Module):

    def __init__(
        self,
        embedding_dim,
        head_dim,
        context_length,
        dropout
    ):
        super().__init__()

        self.key = nn.Linear(
            embedding_dim,
            head_dim,
            bias=False
        )

        self.query = nn.Linear(
            embedding_dim,
            head_dim,
            bias=False
        )

        self.value = nn.Linear(
            embedding_dim,
            head_dim,
            bias=False
        )

        self.dropout = nn.Dropout(dropout)

        self.register_buffer(
            "tril",
            torch.tril(
                torch.ones(
                    context_length,
                    context_length
                )
            )
        )

    def forward(self, x):

        B, T, C = x.shape

        k = self.key(x)
        q = self.query(x)

        attention_scores = q @ k.transpose(-2, -1)

        attention_scores = (
            attention_scores
            * (k.shape[-1] ** -0.5)
        )

        attention_scores = attention_scores.masked_fill(
            self.tril[:T, :T] == 0,
            float("-inf")
        )

        attention_weights = F.softmax(
            attention_scores,
            dim=-1
        )

        attention_weights = self.dropout(
            attention_weights
        )

        v = self.value(x)

        return attention_weights @ v

In [11]:
class MultiHeadAttention(nn.Module):

    def __init__(
        self,
        embedding_dim,
        num_heads,
        context_length,
        dropout
    ):
        super().__init__()

        head_dim = embedding_dim // num_heads

        self.heads = nn.ModuleList([
            Head(
                embedding_dim,
                head_dim,
                context_length,
                dropout
            )
            for _ in range(num_heads)
        ])

        self.projection = nn.Linear(
            embedding_dim,
            embedding_dim
        )

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):

        outputs = [
            head(x)
            for head in self.heads
        ]

        x = torch.cat(
            outputs,
            dim=-1
        )

        x = self.projection(x)

        return self.dropout(x)

In [12]:
class FeedForward(nn.Module):

    def __init__(
        self,
        embedding_dim,
        dropout
    ):
        super().__init__()

        self.network = nn.Sequential(

            nn.Linear(
                embedding_dim,
                4 * embedding_dim
            ),

            nn.GELU(),

            nn.Linear(
                4 * embedding_dim,
                embedding_dim
            ),

            nn.Dropout(dropout)
        )

    def forward(self, x):
        return self.network(x)

In [13]:
class TransformerBlock(nn.Module):

    def __init__(
        self,
        embedding_dim,
        num_heads,
        context_length,
        dropout
    ):
        super().__init__()

        self.ln1 = nn.LayerNorm(
            embedding_dim
        )

        self.attention = MultiHeadAttention(
            embedding_dim,
            num_heads,
            context_length,
            dropout
        )

        self.ln2 = nn.LayerNorm(
            embedding_dim
        )

        self.ffn = FeedForward(
            embedding_dim,
            dropout
        )

    def forward(self, x):

        x = x + self.attention(
            self.ln1(x)
        )

        x = x + self.ffn(
            self.ln2(x)
        )

        return x

In [14]:
class MiniGPT(nn.Module):

    def __init__(
        self,
        vocab_size,
        context_length,
        embedding_dim,
        num_heads,
        num_layers,
        dropout
    ):
        super().__init__()

        self.context_length = context_length

        self.token_embedding = nn.Embedding(
            vocab_size,
            embedding_dim
        )

        self.position_embedding = nn.Embedding(
            context_length,
            embedding_dim
        )

        self.blocks = nn.Sequential(
            *[
                TransformerBlock(
                    embedding_dim,
                    num_heads,
                    context_length,
                    dropout
                )
                for _ in range(num_layers)
            ]
        )

        self.ln_final = nn.LayerNorm(
            embedding_dim
        )

        self.lm_head = nn.Linear(
            embedding_dim,
            vocab_size
        )

    def forward(self, idx):

        B, T = idx.shape

        token_embeddings = self.token_embedding(idx)

        positions = torch.arange(
            T,
            device=idx.device
        )

        position_embeddings = self.position_embedding(
            positions
        )

        x = token_embeddings + position_embeddings

        x = self.blocks(x)

        x = self.ln_final(x)

        logits = self.lm_head(x)

        return logits

In [15]:
model = MiniGPT(
    vocab_size=vocab_size,
    context_length=context_length,
    embedding_dim=embedding_dim,
    num_heads=num_heads,
    num_layers=num_layers,
    dropout=dropout
).to(device)

model.load_state_dict(
    checkpoint["model_state_dict"]
)

model.eval()

print("Model loaded successfully.")

Model loaded successfully.


In [16]:
total_parameters = sum(
    p.numel()
    for p in model.parameters()
)

trainable_parameters = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print("=" * 50)
print("MODEL INFORMATION")
print("=" * 50)

print(f"Total parameters     : {total_parameters:,}")
print(f"Trainable parameters : {trainable_parameters:,}")
print(f"Embedding dimension  : {embedding_dim}")
print(f"Transformer layers   : {num_layers}")
print(f"Attention heads      : {num_heads}")
print(f"Context length       : {context_length}")
print(f"Vocabulary size      : {vocab_size}")

MODEL INFORMATION
Total parameters     : 3,221,567
Trainable parameters : 3,221,567
Embedding dimension  : 256
Transformer layers   : 4
Attention heads      : 4
Context length       : 128
Vocabulary size      : 63


In [17]:
DATA_PATH = "../data/processed/tiny-shakespeare_clean.txt"

with open(
    DATA_PATH,
    "r",
    encoding="utf-8"
) as file:

    text = file.read()

encoded_text = torch.tensor(
    encode(text),
    dtype=torch.long
)

split_index = int(
    0.9 * len(encoded_text)
)

train_data = encoded_text[:split_index]

val_data = encoded_text[split_index:]

print("Total tokens :", len(encoded_text))
print("Train tokens :", len(train_data))
print("Val tokens   :", len(val_data))

Total tokens : 1114961
Train tokens : 1003464
Val tokens   : 111497


In [18]:
def get_batch(
    data,
    batch_size=32,
    context_length=128
):

    starts = torch.randint(
        0,
        len(data) - context_length - 1,
        (batch_size,)
    )

    x = torch.stack([
        data[start:start + context_length]
        for start in starts
    ])

    y = torch.stack([
        data[start + 1:start + context_length + 1]
        for start in starts
    ])

    return x, y

In [19]:
@torch.no_grad()
def estimate_loss(
    model,
    data,
    batches=100
):

    model.eval()

    losses = []

    for _ in range(batches):

        x, y = get_batch(
            data,
            batch_size=32,
            context_length=context_length
        )

        x = x.to(device)
        y = y.to(device)

        logits = model(x)

        loss = F.cross_entropy(
            logits.view(-1, vocab_size),
            y.view(-1)
        )

        losses.append(
            loss.item()
        )

    return sum(losses) / len(losses)

In [20]:
train_loss = estimate_loss(
    model,
    train_data
)

val_loss = estimate_loss(
    model,
    val_data
)

print("=" * 50)
print("FINAL MODEL EVALUATION")
print("=" * 50)

print(f"Training Loss   : {train_loss:.4f}")
print(f"Validation Loss : {val_loss:.4f}")

FINAL MODEL EVALUATION
Training Loss   : 1.3509
Validation Loss : 1.5692


In [21]:
train_perplexity = math.exp(train_loss)
val_perplexity = math.exp(val_loss)

print("=" * 50)
print("PERPLEXITY")
print("=" * 50)

print(f"Training Perplexity   : {train_perplexity:.2f}")
print(f"Validation Perplexity : {val_perplexity:.2f}")

PERPLEXITY
Training Perplexity   : 3.86
Validation Perplexity : 4.80


In [22]:
@torch.no_grad()
def calculate_accuracy(
    model,
    data,
    batches=100
):

    model.eval()

    correct = 0
    total = 0

    for _ in range(batches):

        x, y = get_batch(
            data,
            batch_size=32,
            context_length=context_length
        )

        x = x.to(device)
        y = y.to(device)

        logits = model(x)

        predictions = torch.argmax(
            logits,
            dim=-1
        )

        correct += (
            predictions == y
        ).sum().item()

        total += y.numel()

    return correct / total

In [23]:
train_accuracy = calculate_accuracy(
    model,
    train_data
)

val_accuracy = calculate_accuracy(
    model,
    val_data
)

print("=" * 50)
print("CHARACTER PREDICTION ACCURACY")
print("=" * 50)

print(
    f"Training Accuracy   : {train_accuracy * 100:.2f}%"
)

print(
    f"Validation Accuracy : {val_accuracy * 100:.2f}%"
)

CHARACTER PREDICTION ACCURACY
Training Accuracy   : 58.17%
Validation Accuracy : 53.52%


In [24]:
print("=" * 60)
print("MINIGPT BASELINE EVALUATION")
print("=" * 60)

print(f"Parameters          : {total_parameters:,}")
print(f"Training Loss       : {train_loss:.4f}")
print(f"Validation Loss     : {val_loss:.4f}")
print(f"Training Perplexity : {train_perplexity:.2f}")
print(f"Val Perplexity      : {val_perplexity:.2f}")
print(f"Training Accuracy   : {train_accuracy * 100:.2f}%")
print(f"Val Accuracy        : {val_accuracy * 100:.2f}%")

print("=" * 60)

MINIGPT BASELINE EVALUATION
Parameters          : 3,221,567
Training Loss       : 1.3509
Validation Loss     : 1.5692
Training Perplexity : 3.86
Val Perplexity      : 4.80
Training Accuracy   : 58.17%
Val Accuracy        : 53.52%


In [25]:
@torch.no_grad()
def generate(
    model,
    prompt,
    max_new_tokens=300,
    temperature=0.8,
    top_k=None
):
    model.eval()

    tokens = encode(prompt)

    idx = torch.tensor(
        [tokens],
        dtype=torch.long,
        device=device
    )

    for _ in range(max_new_tokens):

        # Keep only the most recent context
        idx_cond = idx[:, -context_length:]

        # Model prediction
        logits = model(idx_cond)

        # Get predictions for the last position
        logits = logits[:, -1, :]

        # Temperature
        logits = logits / temperature

        # Optional Top-K sampling
        if top_k is not None:
            values, _ = torch.topk(
                logits,
                min(top_k, logits.size(-1))
            )

            logits[
                logits < values[:, [-1]]
            ] = float("-inf")

        # Convert to probabilities
        probabilities = F.softmax(
            logits,
            dim=-1
        )

        # Sample next token
        next_token = torch.multinomial(
            probabilities,
            num_samples=1
        )

        # Add token
        idx = torch.cat(
            (idx, next_token),
            dim=1
        )

    return decode(
        idx[0].tolist()
    )

In [26]:
prompt = "ROMEO:"

generated = generate(
    model,
    prompt,
    max_new_tokens=500,
    temperature=0.8
)

print(generated)

ROMEO:
Your soul asks and say made so fast; therefore,
What we do you are as a hold to make how.

LUCIO:
Well do not be not more.

First Citizen:
Here's needs in the compliantion broke;
And thou art unto my suit all hearts up the fair
That no who mark in a sever amend;
For a hearts it show upon thy souls,
And they were you her, fear they have your comploy
And shame to you in this person the own to royal,
Call to many hand.
My lords, should not I fear them a knee
The lords of thy lamb, and be left him 


In [27]:
prompts = [
    "ROMEO:",
    "HAMLET:",
    "KING:",
    "LORD:",
    "What",
    "To be"
]

for prompt in prompts:

    print("=" * 70)
    print(f"PROMPT: {prompt}")
    print("=" * 70)

    output = generate(
        model,
        prompt,
        max_new_tokens=200,
        temperature=0.8,
        top_k=20
    )

    print(output)
    print()

PROMPT: ROMEO:
ROMEO:
My wrong gods! So dream posts of thy tongue;
And, or never than and accorn to lay,
Where is such the head the great of men
And more in my bosom, if their lives provoke.

MENENIUS:
Sir heart with swea

PROMPT: HAMLET:
HAMLET:
The prizent that let me half buried them;
And he was when we we must not love,
The state of our venour pain is more.

CORIOLANUS:
I did tell of your tenders now,
Sir: I'll thee be shall be alonerable

PROMPT: KING:
KING:
He would did he worse thy dear's daughter;
Halk his requer.

EDWARD:
My lord, not proposed of York, much lives
Let him grown in the pride house, and say your before,
And let's true love your lord,
Wh

PROMPT: LORD:
LORD:
So the greates of our banish is a chorse;
With afford inquirels friends, and me none there
The part day in the world of you know as them
Is thy drum, and part to be a man.

LUCIO:
The fair franketors

PROMPT: What
What souls of my creasure, we shall fight,
But spirity pass the merity of his diath,
Bare my lord

In [28]:
prompt = "ROMEO:"

temperatures = [
    0.5,
    0.8,
    1.0,
    1.2
]

for temperature in temperatures:

    print("=" * 70)
    print(f"TEMPERATURE = {temperature}")
    print("=" * 70)

    output = generate(
        model,
        prompt,
        max_new_tokens=200,
        temperature=temperature,
        top_k=20
    )

    print(output)

TEMPERATURE = 0.5
ROMEO:
What is see he done?

GREMIO:
What may made in the dear words and the death,
Thou art to the flowers in thy bed with here,
Thou art thou art and thy fair lands.

ANTIGONUS:
God the piring thee,
I was
TEMPERATURE = 0.8
ROMEO:
How! then, gave their callains sleeping should us
As so rest honour'd to say thee to the queen;
Outeous call luck, of my oracle.
To true, of I smy hast more which theirs
the golden of her comes out t
TEMPERATURE = 1.0
ROMEO:
O, if it will it.

LEONTES:
Now.
That the sir, for too say fherein my trance,
These hangess been and me sorrow.
All make for their that hunt play wilt.

NORTHUMBERLAND:
Nay, part, whom I have die fai
TEMPERATURE = 1.2
ROMEO:

MEINA:
This own love.

First Gentleman:
I'll be life.

LUCENTIO:
Come, nor savip with since me, save to-night,
May let me I both, sir, one play his devil.

RICHARD:
Bid with hasd is sirrels.
But, th


In [29]:
prompt = "ROMEO:"

top_k_values = [
    3,
    5,
    10,
    20,
    None
]

for k in top_k_values:

    print("=" * 70)

    if k is None:
        print("TOP-K = Disabled")
    else:
        print(f"TOP-K = {k}")

    print("=" * 70)

    output = generate(
        model,
        prompt,
        max_new_tokens=200,
        temperature=0.8,
        top_k=k
    )

    print(output)

TOP-K = 3
ROMEO:
With the markels of his present, and he do not
The seater of the charge of his sister,
The perceives of his senator,
And we shall to heaven and the world and the state,
That have she shame to see the
TOP-K = 5
ROMEO:
Where thou?

MENENIUS:
There's the man?

MARCIUS:
Ha!

COMINIUS:
I have so die a good strange to be so;
The season, all the whole it is the strong,
And sain, if, they we have done, shall be thy sight
TOP-K = 10
ROMEO:
What, let me not struck'd from my mother?

CAMILLO:
What thou?

POMPEY:
Triumph as my lord? I will be a conting to the
stide me the people is business hath done?

CORIOLANUS:
Why, is that is with dri
TOP-K = 20
ROMEO:
More shall be place.

GREMIO:
What's count?

ANGELO:
I say here?

CAMILLO:
Lords and call thee thy air birthrous assure.

SAMPSON:
O, my lord, as he's was a most have
Here done, and so shall I rescor
TOP-K = Disabled
ROMEO:
As I do with a from this land-so your death droff;
For your joyments make they see
Your gainst consul, or 

In [30]:
baseline_results = {
    "parameters": total_parameters,
    "training_loss": train_loss,
    "validation_loss": val_loss,
    "training_perplexity": train_perplexity,
    "validation_perplexity": val_perplexity,
    "training_accuracy": train_accuracy,
    "validation_accuracy": val_accuracy
}

print(baseline_results)

{'parameters': 3221567, 'training_loss': 1.3509242272377013, 'validation_loss': 1.5692376387119293, 'training_perplexity': 3.8609923164462248, 'validation_perplexity': 4.80298518771129, 'training_accuracy': 0.581708984375, 'validation_accuracy': 0.5351708984375}


In [31]:
import json

with open(
    "../models/baseline_evaluation.json",
    "w"
) as file:

    json.dump(
        baseline_results,
        file,
        indent=4
    )

print("Baseline evaluation saved.")

Baseline evaluation saved.
